# Final Results and Documentation

This notebook consolidates the existing ML experiment artifacts. It does not retrain or reevaluate models, modify data or preprocessing, create a split, or perform a new experiment.

In [1]:
from pathlib import Path
import json
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else (cwd / 'pixels-to-water' if (cwd / 'pixels-to-water' / 'results').exists() else cwd)
RESULTS_DIR = PROJECT_ROOT / 'results'
SUMMARY_CSV = RESULTS_DIR / 'final_experiment_summary.csv'
SUMMARY_JSON = RESULTS_DIR / 'final_experiment_summary.json'
print(f'Project root: {PROJECT_ROOT}')

Project root: c:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water


In [2]:
with open(RESULTS_DIR / 'resnet18_test_metrics.json', encoding='utf-8') as file:
    resnet_metrics = json.load(file)
with open(RESULTS_DIR / 'efficientnet_b0_test_metrics.json', encoding='utf-8') as file:
    efficientnet_metrics = json.load(file)
with open(RESULTS_DIR / 'final_model_selection.json', encoding='utf-8') as file:
    selection = json.load(file)
comparison = pd.read_csv(RESULTS_DIR / 'model_comparison.csv')
error_summary = pd.read_csv(RESULTS_DIR / 'error_analysis_summary.csv')
test_manifest = pd.read_csv(RESULTS_DIR / 'test.csv')

metric_map = {
    'Accuracy': 'test_accuracy',
    'Precision': 'precision',
    'Recall': 'recall',
    'F1-score': 'f1_score',
    'Top-5 Accuracy': 'top5_accuracy',
}
summary = pd.DataFrame([
    {'Model': 'ResNet-18', **{label: resnet_metrics[key] for label, key in metric_map.items()}},
    {'Model': 'EfficientNet-B0', **{label: efficientnet_metrics[key] for label, key in metric_map.items()}},
])
comparison_values = comparison.set_index('Metric')
for label in metric_map:
    assert summary.loc[0, label] == comparison_values.loc[('Test Accuracy' if label == 'Accuracy' else label), 'ResNet-18']
    assert summary.loc[1, label] == comparison_values.loc[('Test Accuracy' if label == 'Accuracy' else label), 'EfficientNet-B0']
print(summary.to_string(index=False))
print(f'Test set size: {len(test_manifest)}')

          Model  Accuracy  Precision  Recall  F1-score  Top-5 Accuracy
      ResNet-18     0.686   0.725493   0.686  0.679440           0.904
EfficientNet-B0     0.742   0.779000   0.742  0.737577           0.928
Test set size: 500


In [3]:
error_counts = error_summary.groupby('model')['number_of_errors'].sum().to_dict()
error_analysis = {
    'ResNet-18 errors': int(error_counts['ResNet-18']),
    'EfficientNet-B0 errors': int(error_counts['EfficientNet-B0']),
    'EfficientNet-B0 fewer errors by': int(error_counts['ResNet-18'] - error_counts['EfficientNet-B0']),
}
statistical_reliability = selection['statistical_reliability_result']
print('Selected model:', selection['selected_model'])
print('Statistical reliability:', statistical_reliability)
print('Error analysis:', error_analysis)
print('Existing visualization artifacts:', [
    path.name for path in [
        RESULTS_DIR / 'model_comparison_summary.png',
        RESULTS_DIR / 'model_parameter_comparison.png',
    ] if path.exists()
])

Selected model: EfficientNet-B0
Statistical reliability: {'test': "McNemar's test", 'alpha': 0.05, 'paired_counts': 'unavailable', 'statistic': None, 'p_value': None, 'conclusion': 'The test was not calculated because aligned per-image predictions for both models are not saved.', 'missing_information': 'One correctness result per test image for both models, aligned by image.'}
Error analysis: {'ResNet-18 errors': 157, 'EfficientNet-B0 errors': 129, 'EfficientNet-B0 fewer errors by': 28}
Existing visualization artifacts: ['model_comparison_summary.png', 'model_parameter_comparison.png']


In [4]:
limitations = [
    'The local dataset is a 5000-image subset of the full Sorghum-100 dataset.',
    'The test set contains 500 images, with 5 images per cultivar, so per-class evaluation metrics have high variance.',
    'McNemar\'s paired comparison was not calculated because aligned per-image predictions for both models are not saved.',
]
conclusion = (
    'The project tested ResNet-18 and EfficientNet-B0 for 100-class sorghum cultivar classification using the same '
    'existing dataset split and preprocessing. EfficientNet-B0 was selected as the final candidate because it had '
    'higher accuracy, precision, recall, F1-score, and top-5 accuracy, and fewer recorded test errors. The models '
    'were compared using existing aggregate metrics and error-analysis results; a paired McNemar test could not be '
    'performed because aligned per-image predictions were unavailable. The main evaluation limitation is the 5-image-per-cultivar '
    'test set, which makes per-class metrics high variance. The next practical step is to carry the selected '
    'EfficientNet-B0 checkpoint into application integration, without claiming universal superiority.'
)
final_summary = {
    'selected_model': selection['selected_model'],
    'key_metrics': summary.set_index('Model').to_dict(),
    'statistical_reliability': statistical_reliability,
    'limitations': limitations,
    'conclusion': conclusion,
}
summary.to_csv(SUMMARY_CSV, index=False)
with open(SUMMARY_JSON, 'w', encoding='utf-8') as file:
    json.dump(final_summary, file, indent=2)

print('Final selected model:', final_summary['selected_model'])
print('Conclusion:', conclusion)
print('Saved:', SUMMARY_CSV)
print('Saved:', SUMMARY_JSON)

Final selected model: EfficientNet-B0
Conclusion: The project tested ResNet-18 and EfficientNet-B0 for 100-class sorghum cultivar classification using the same existing dataset split and preprocessing. EfficientNet-B0 was selected as the final candidate because it had higher accuracy, precision, recall, F1-score, and top-5 accuracy, and fewer recorded test errors. The models were compared using existing aggregate metrics and error-analysis results; a paired McNemar test could not be performed because aligned per-image predictions were unavailable. The main evaluation limitation is the 5-image-per-cultivar test set, which makes per-class metrics high variance. The next practical step is to carry the selected EfficientNet-B0 checkpoint into application integration, without claiming universal superiority.
Saved: c:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water\results\final_experiment_summary.csv
Saved: c:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water\results